# TransitStats — Prediction Engine V4
Logistic regression route classifier trained on trip history.
Runs alongside V3 — does not touch the app.

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('trips.csv')
print(f'{len(df)} trips loaded')
df.head()

## 1. Clean & Normalize

In [ ]:
def base_route(r):
    """Strip suffix variants so 510a/510A/510b all become 510."""
    return re.sub(r'[a-zA-Z]+$', '', str(r).strip()).strip()

df['route_base'] = df['route'].apply(base_route)
df['start_time'] = pd.to_datetime(df['start_time'], format='ISO8601', utc=True)

# Drop rows missing the essentials
df = df.dropna(subset=['route_base', 'start_stop', 'hour_of_day', 'day_of_week'])

# Keep only routes with enough trips to train on (filters garbage + rare routes)
route_counts = df['route_base'].value_counts()
df = df[df['route_base'].isin(route_counts[route_counts >= 3].index)]

print('Route distribution after normalization:')
print(df['route_base'].value_counts().head(15))
print(f'\n{len(df)} trips kept')

## 2. Feature Engineering

In [ ]:
# Cyclical encoding for hour — so 23:00 and 0:00 are close together
df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)

# Cyclical encoding for day of week
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Encode start stop as a category
df['start_stop_clean'] = df['start_stop'].str.strip().str.lower()
stop_dummies = pd.get_dummies(df['start_stop_clean'], prefix='stop')

features = pd.concat([
    df[['hour_sin', 'hour_cos', 'day_sin', 'day_cos']],
    stop_dummies
], axis=1)

labels = df['route_base']

print(f'Feature matrix: {features.shape[0]} rows × {features.shape[1]} columns')
print(f'Unique routes (classes): {labels.nunique()}')

## 3. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, top_k_accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Train: {len(X_train)}  Test: {len(X_test)}')

## 4. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

top1 = (y_pred == y_test).mean()
top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=model.classes_)

print(f'Top-1 accuracy: {top1:.1%}  (correct first guess)')
print(f'Top-3 accuracy: {top3:.1%}  (correct answer in top 3)')
print()
print(classification_report(y_test, y_pred))

## 4b. Train XGBoost (V5 Benchmark)

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost needs integer labels — fit on full label set so all classes are known
le = LabelEncoder()
le.fit(labels)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0,
)
xgb.fit(X_train, y_train_enc)

y_pred_xgb  = le.inverse_transform(xgb.predict(X_test))
y_proba_xgb = xgb.predict_proba(X_test)

top1_xgb = (y_pred_xgb == y_test.values).mean()
# labels= required because test split may be missing some classes
top3_xgb = top_k_accuracy_score(y_test_enc, y_proba_xgb, k=3,
                                  labels=list(range(len(le.classes_))))

print('=== V4 vs V5 Benchmark ===')
print(f'V4 Logistic Regression — Top-1: {top1:.1%}   Top-3: {top3:.1%}')
print(f'V5 XGBoost            — Top-1: {top1_xgb:.1%}   Top-3: {top3_xgb:.1%}')
delta1 = (top1_xgb - top1) * 100
delta3 = (top3_xgb - top3) * 100
print(f'Delta                 — Top-1: {delta1:+.1f}pp  Top-3: {delta3:+.1f}pp')

## 5. What Did the Model Learn?

In [ ]:
# Which features matter most for each route?
feature_names = features.columns.tolist()

for i, route in enumerate(model.classes_):
    coefs = model.coef_[i]
    top_idx = np.argsort(np.abs(coefs))[-5:][::-1]
    top_feats = [(feature_names[j], round(coefs[j], 3)) for j in top_idx]
    print(f'Route {route}: {top_feats}')

## 6. Make a Prediction

In [ ]:
def predict_route(stop_name, hour, day_of_week, top_n=3):
    """Predict likely routes given a stop, hour, and day."""
    row = pd.DataFrame([{
        'hour_sin': np.sin(2 * np.pi * hour / 24),
        'hour_cos': np.cos(2 * np.pi * hour / 24),
        'day_sin':  np.sin(2 * np.pi * day_of_week / 7),
        'day_cos':  np.cos(2 * np.pi * day_of_week / 7),
    }])

    # Add stop columns (all zero, then set the matching one)
    for col in stop_dummies.columns:
        row[col] = 0
    stop_col = f'stop_{stop_name.strip().lower()}'
    if stop_col in row.columns:
        row[stop_col] = 1
    else:
        print(f'(Stop "{stop_name}" not seen in training — time/day features only)')

    row = row[features.columns]  # ensure column order matches
    proba = model.predict_proba(row)[0]
    top_idx = np.argsort(proba)[-top_n:][::-1]

    print(f'Stop: {stop_name}  |  Hour: {hour:02d}:00  |  Day: {["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][day_of_week]}')
    for idx in top_idx:
        print(f'  Route {model.classes_[idx]:>5}  {proba[idx]:.1%}')

# Try it — change these to match your actual stops
predict_route('york university', hour=8, day_of_week=0)   # Monday 8am
print()
predict_route('spadina station', hour=17, day_of_week=1)  # Tuesday 5pm

## 7. Export Weights for Cloud Function
Once you're happy with accuracy, run this to save the model.

In [ ]:
import json

export = {
    'classes': model.classes_.tolist(),
    'coef': model.coef_.tolist(),
    'intercept': model.intercept_.tolist(),
    'feature_names': feature_names,
    'stop_columns': stop_dummies.columns.tolist(),
}

with open('model_v4.json', 'w') as f:
    json.dump(export, f)

print('Saved model_v4.json')